In [7]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import joblib
import os

In [8]:
class RiskEngine:
    def __init__(self, model_path="models/risk_model.json"):
        self.model_path = model_path
        self.model = xgb.XGBClassifier()
        self.encoders = {}
        self.features = ['carrier', 'weather', 'warehouse_load_pct', 'traffic_density']

    def _prepare_data(self, df, training=False):
        """Encodes categorical strings into numeric values."""
        df_copy = df.copy()
        for col in ['carrier', 'weather']:
            if training:
                le = LabelEncoder()
                df_copy[col] = le.fit_transform(df_copy[col])
                self.encoders[col] = le
            else:
                # Use existing encoder for live inference
                df_copy[col] = self.encoders[col].transform(df_copy[col])
        return df_copy[self.features]

    def train(self, data_path="historical_logistics.csv"):
        """Trains the XGBoost model on historical patterns."""
        df = pd.read_csv(data_path)
        X = self._prepare_data(df, training=True)
        y = df['is_delayed']
        
        # Train XGBoost
        self.model.fit(X, y)
        
        # Save encoders and model
        os.makedirs("models", exist_ok=True)
        self.model.save_model(self.model_path)
        joblib.dump(self.encoders, "models/encoders.joblib")
        print(f"Risk Engine trained and saved to {self.model_path}")

    def load_engine(self):
        """Loads the model and encoders for the agent to use."""
        self.model.load_model(self.model_path)
        self.encoders = joblib.load("models/encoders.joblib")

    def get_delay_likelihood(self, live_data):
        """
        Takes a single shipment dict and returns probability (0.0 - 1.0).
        Used by the Reasoning Agent.
        """
        df_input = pd.DataFrame([live_data])
        X_live = self._prepare_data(df_input, training=False)
        
        # Returns [prob_on_time, prob_delayed]
        probs = self.model.predict_proba(X_live)
        return float(probs[0][1])